# 2. Evaluation

Test thử train trên nhiều model

In [1]:
import os
import random
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# If this import fails, run in a notebook cell: %pip install wandb
import wandb

try:
    import pandas as pd
except ImportError:
    pd = None

- Hyperparameters in `wandb.config`
- Train loss and accuracy
- Evaluation loss, accuracy, macro precision, macro recall, and macro F1
- Confusion matrix
- Per-class evaluation table
- Prediction examples with images
- Model checkpoint artifact

## Project setup

In [2]:
PROJECT_NAME = "mnist-wandb-evaluation-v1"
ENTITY = None  # Example: "your-wandb-username". Keep None to use your default entity.

DATA_DIR = Path("data")
CHECKPOINT_DIR = Path("ckpts")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = [str(i) for i in range(10)]

# Uncomment this if you want local-only runs, then sync later with: wandb sync wandb/offline-*
# os.environ["WANDB_MODE"] = "offline"

print(f"Using device: {DEVICE}")
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Mario\_netrc.


Using device: cuda


wandb: Currently logged in as: zintom69 (zintomfromnowhere) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Hyperparameter configurations

In [ ]:
BASE_CONFIG = {
    "dataset": "MNIST",
    "epochs": 10,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "optimizer": "adam",
    "weight_decay": 0.0,
    "momentum": 0.9,
    "dropout": 0.0,
    "train_subset": 12000, # train nhỏ thôi
    "test_subset": 2000,
    "log_every_n_batches": 50,
    "num_workers": 0,
    "seed": 42,
}


def make_config(**overrides):
    config = deepcopy(BASE_CONFIG)
    config.update(overrides)
    return config


BASELINE_CONFIGS = [
    make_config(
        model_name="mlp_small",
        learning_rate=1e-3,
        optimizer="adam",
        dropout=0.0,
        seed=101,
    ),
    make_config(
        model_name="mlp_deep",
        learning_rate=3e-4,
        optimizer="adamw",
        dropout=0.25,
        weight_decay=1e-4,
        seed=102,
    ),
    make_config(
        model_name="tiny_cnn",
        learning_rate=1e-3,
        optimizer="adam",
        dropout=0.0,
        seed=103,
    ),
    make_config(
        model_name="dropout_cnn",
        learning_rate=1e-3,
        optimizer="adam",
        dropout=0.35,
        weight_decay=1e-5,
        seed=104,
    ),
    make_config(
        model_name="batchnorm_cnn",
        learning_rate=1e-2,
        optimizer="sgd",
        dropout=0.15,
        momentum=0.9,
        weight_decay=1e-4,
        seed=105,
    ),
]

BASELINE_CONFIGS

[{'dataset': 'MNIST',
  'epochs': 10,
  'batch_size': 128,
  'learning_rate': 0.001,
  'optimizer': 'adam',
  'weight_decay': 0.0,
  'momentum': 0.9,
  'dropout': 0.0,
  'train_subset': 12000,
  'test_subset': 2000,
  'log_every_n_batches': 50,
  'num_workers': 0,
  'seed': 101,
  'model_name': 'mlp_small'},
 {'dataset': 'MNIST',
  'epochs': 10,
  'batch_size': 128,
  'learning_rate': 0.0003,
  'optimizer': 'adamw',
  'weight_decay': 0.0001,
  'momentum': 0.9,
  'dropout': 0.25,
  'train_subset': 12000,
  'test_subset': 2000,
  'log_every_n_batches': 50,
  'num_workers': 0,
  'seed': 102,
  'model_name': 'mlp_deep'},
 {'dataset': 'MNIST',
  'epochs': 10,
  'batch_size': 128,
  'learning_rate': 0.001,
  'optimizer': 'adam',
  'weight_decay': 0.0,
  'momentum': 0.9,
  'dropout': 0.0,
  'train_subset': 12000,
  'test_subset': 2000,
  'log_every_n_batches': 50,
  'num_workers': 0,
  'seed': 103,
  'model_name': 'tiny_cnn'},
 {'dataset': 'MNIST',
  'epochs': 10,
  'batch_size': 128,
  'lear

## Data helpers

In [5]:
MNIST_MEAN = 0.1307
MNIST_STD = 0.3081


def set_seed(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def subset_dataset(dataset, subset_size, seed):
    if subset_size is None:
        return dataset
    subset_size = int(subset_size)
    if subset_size <= 0 or subset_size >= len(dataset):
        return dataset

    generator = torch.Generator().manual_seed(int(seed))
    indices = torch.randperm(len(dataset), generator=generator)[:subset_size].tolist()
    return Subset(dataset, indices)


def get_dataloaders(config):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
    ])

    train_dataset = datasets.MNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=transform,
    )
    test_dataset = datasets.MNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=transform,
    )

    train_dataset = subset_dataset(train_dataset, config.get("train_subset"), config["seed"])
    test_dataset = subset_dataset(test_dataset, config.get("test_subset"), config["seed"] + 1000)

    generator = torch.Generator().manual_seed(int(config["seed"]))
    train_loader = DataLoader(
        train_dataset,
        batch_size=int(config["batch_size"]),
        shuffle=True,
        num_workers=int(config.get("num_workers", 0)),
        generator=generator,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=int(config["batch_size"]),
        shuffle=False,
        num_workers=int(config.get("num_workers", 0)),
    )

    return train_loader, test_loader


def denormalize_mnist(batch):
    return (batch * MNIST_STD + MNIST_MEAN).clamp(0, 1)

## Five model architectures

In [6]:
class MLPSmall(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)


class MLPDeep(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


class TinyCNN(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(32 * 7 * 7, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


class DropoutCNN(nn.Module):
    def __init__(self, dropout=0.35):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


class BatchNormCNN(nn.Module):
    def __init__(self, dropout=0.15):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


MODEL_BUILDERS = {
    "mlp_small": MLPSmall,
    "mlp_deep": MLPDeep,
    "tiny_cnn": TinyCNN,
    "dropout_cnn": DropoutCNN,
    "batchnorm_cnn": BatchNormCNN,
}


def build_model(config):
    model_name = config["model_name"]
    if model_name not in MODEL_BUILDERS:
        raise ValueError(f"Unknown model_name={model_name}. Options: {list(MODEL_BUILDERS)}")
    return MODEL_BUILDERS[model_name](dropout=float(config.get("dropout", 0.0))).to(DEVICE)

## Training and evaluation helpers

In [7]:
def build_optimizer(model, config):
    optimizer_name = str(config["optimizer"]).lower()
    lr = float(config["learning_rate"])
    weight_decay = float(config.get("weight_decay", 0.0))

    if optimizer_name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if optimizer_name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    if optimizer_name == "sgd":
        return torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=float(config.get("momentum", 0.0)),
            weight_decay=weight_decay,
        )

    raise ValueError(f"Unknown optimizer={optimizer_name}")


def compute_classification_metrics(y_true, y_pred, num_classes=10):
    y_true = torch.as_tensor(y_true, dtype=torch.long)
    y_pred = torch.as_tensor(y_pred, dtype=torch.long)
    confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)

    for target, pred in zip(y_true, y_pred):
        confusion[target, pred] += 1

    eps = 1e-12
    tp = confusion.diag().float()
    precision = tp / (confusion.sum(dim=0).float() + eps)
    recall = tp / (confusion.sum(dim=1).float() + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    accuracy = tp.sum() / confusion.sum().float().clamp_min(1.0)

    metrics = {
        "accuracy": accuracy.item(),
        "precision_macro": precision.mean().item(),
        "recall_macro": recall.mean().item(),
        "f1_macro": f1.mean().item(),
    }
    per_class_rows = []
    for idx, class_name in enumerate(CLASSES):
        support = int(confusion[idx].sum().item())
        per_class_rows.append([
            class_name,
            support,
            precision[idx].item(),
            recall[idx].item(),
            f1[idx].item(),
        ])

    return metrics, confusion, per_class_rows


def train_one_epoch(model, dataloader, loss_fn, optimizer, epoch_index, config):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0
    log_every = int(config.get("log_every_n_batches", 50))
    num_batches = len(dataloader)

    for batch_index, (inputs, targets) in enumerate(dataloader):
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = loss_fn(logits, targets)
        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)
        predictions = logits.argmax(dim=1)
        batch_correct = (predictions == targets).sum().item()
        batch_loss = loss.item()

        total_loss += batch_loss * batch_size
        total_correct += batch_correct
        total_seen += batch_size

        if batch_index % log_every == 0:
            batch_step = epoch_index * num_batches + batch_index + 1
            wandb.log({
                "batch_step": batch_step,
                "batch/loss": batch_loss,
                "batch/accuracy": batch_correct / batch_size,
                "batch/learning_rate": optimizer.param_groups[0]["lr"],
            })

    return {
        "loss": total_loss / max(total_seen, 1),
        "accuracy": total_correct / max(total_seen, 1),
    }


@torch.no_grad()
def evaluate(model, dataloader, loss_fn, collect_examples=False, max_examples=32):
    model.eval()
    total_loss = 0.0
    total_seen = 0
    all_targets = []
    all_predictions = []
    example_rows = []

    for inputs, targets in dataloader:
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)

        logits = model(inputs)
        loss = loss_fn(logits, targets)
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        total_seen += batch_size
        all_targets.extend(targets.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())

        if collect_examples and len(example_rows) < max_examples:
            remaining = max_examples - len(example_rows)
            take = min(remaining, batch_size)
            images = denormalize_mnist(inputs[:take].detach().cpu())
            for i in range(take):
                true_label = int(targets[i].item())
                pred_label = int(predictions[i].item())
                confidence = float(probabilities[i, pred_label].item())
                example_rows.append([
                    wandb.Image(
                        images[i].squeeze(0).numpy(),
                        caption=f"pred={pred_label}, true={true_label}",
                    ),
                    true_label,
                    pred_label,
                    confidence,
                    bool(true_label == pred_label),
                ])

    metrics, confusion, per_class_rows = compute_classification_metrics(all_targets, all_predictions)
    metrics["loss"] = total_loss / max(total_seen, 1)

    return metrics, confusion, per_class_rows, all_targets, all_predictions, example_rows

## Experiment runner

In [10]:
def make_run_name(config):
    return (
        f"{config['model_name']}-"
        f"{config['optimizer']}-"
        f"lr{float(config['learning_rate']):.0e}-"
        f"bs{int(config['batch_size'])}-"
        f"drop{float(config.get('dropout', 0.0)):.2f}"
    )


def define_wandb_metrics():
    wandb.define_metric("epoch")
    wandb.define_metric("batch_step")
    wandb.define_metric("batch/*", step_metric="batch_step")
    wandb.define_metric("train/*", step_metric="epoch")
    wandb.define_metric("eval/*", step_metric="epoch")


def log_final_evaluation_artifacts(config, metrics, confusion, per_class_rows, y_true, y_pred, example_rows):
    per_class_table = wandb.Table(
        columns=["class", "support", "precision", "recall", "f1"],
        data=per_class_rows,
    )
    examples_table = wandb.Table(
        columns=["image", "true_label", "pred_label", "confidence", "correct"],
        data=example_rows,
    )

    wandb.log({
        "epoch": int(config["epochs"]),
        "eval/confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_true,
            preds=y_pred,
            class_names=CLASSES,
        ),
        "eval/per_class_metrics": per_class_table,
        "eval/prediction_examples": examples_table,
        "eval/final_loss": metrics["loss"],
        "eval/final_accuracy": metrics["accuracy"],
        "eval/final_f1_macro": metrics["f1_macro"],
    })


def log_model_artifact(run, model, config):
    checkpoint_path = CHECKPOINT_DIR / f"{run.id}_{config['model_name']}.pt"
    torch.save({
        "model_state_dict": model.state_dict(),
        "config": dict(config),
        "classes": CLASSES,
    }, checkpoint_path)

    artifact = wandb.Artifact(
        name=f"mnist-{config['model_name']}-{run.id}",
        type="model",
        metadata=dict(config),
    )
    artifact.add_file(str(checkpoint_path))
    run.log_artifact(artifact)
    return checkpoint_path


def run_experiment(config=None, group="mnist-5-baselines", job_type="train-eval", from_sweep=False):
    base_config = deepcopy(BASE_CONFIG)
    if config is not None:
        base_config.update(config)

    init_kwargs = {
        "project": PROJECT_NAME,
        "entity": ENTITY,
        "config": base_config,
        "group": group,
        "job_type": job_type,
        "save_code": True,
        "reinit": "finish_previous",
    }
    if not from_sweep:
        init_kwargs["name"] = make_run_name(base_config)

    run = wandb.init(**init_kwargs)
    config = dict(wandb.config)
    run.name = make_run_name(config)
    # run.save()

    try:
        define_wandb_metrics()
        set_seed(config["seed"])

        train_loader, test_loader = get_dataloaders(config)
        model = build_model(config)
        loss_fn = nn.CrossEntropyLoss()
        optimizer = build_optimizer(model, config)

        wandb.watch(model, criterion=loss_fn, log="all", log_freq=100)

        best_eval_accuracy = 0.0
        start_time = time.time()

        for epoch in range(int(config["epochs"])):
            train_metrics = train_one_epoch(model, train_loader, loss_fn, optimizer, epoch, config)
            eval_metrics, _, _, _, _, _ = evaluate(model, test_loader, loss_fn)
            best_eval_accuracy = max(best_eval_accuracy, eval_metrics["accuracy"])

            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "eval/loss": eval_metrics["loss"],
                "eval/accuracy": eval_metrics["accuracy"],
                "eval/precision_macro": eval_metrics["precision_macro"],
                "eval/recall_macro": eval_metrics["recall_macro"],
                "eval/f1_macro": eval_metrics["f1_macro"],
                "eval/best_accuracy": best_eval_accuracy,
            })

            print(
                f"{run.name} | epoch {epoch + 1}/{config['epochs']} | "
                f"train_acc={train_metrics['accuracy']:.4f} | "
                f"eval_acc={eval_metrics['accuracy']:.4f} | "
                f"eval_loss={eval_metrics['loss']:.4f}"
            )

        final_metrics, confusion, per_class_rows, y_true, y_pred, example_rows = evaluate(
            model,
            test_loader,
            loss_fn,
            collect_examples=True,
            max_examples=32,
        )
        log_final_evaluation_artifacts(
            config,
            final_metrics,
            confusion,
            per_class_rows,
            y_true,
            y_pred,
            example_rows,
        )
        checkpoint_path = log_model_artifact(run, model, config)

        elapsed_seconds = time.time() - start_time
        run.summary["best_eval_accuracy"] = best_eval_accuracy
        run.summary["final_eval_accuracy"] = final_metrics["accuracy"]
        run.summary["final_eval_f1_macro"] = final_metrics["f1_macro"]
        run.summary["elapsed_seconds"] = elapsed_seconds
        run.summary["checkpoint_path"] = str(checkpoint_path)

        return {
            "run_id": run.id,
            "run_name": run.name,
            "model_name": config["model_name"],
            "optimizer": config["optimizer"],
            "learning_rate": config["learning_rate"],
            "batch_size": config["batch_size"],
            "dropout": config.get("dropout", 0.0),
            "best_eval_accuracy": best_eval_accuracy,
            "final_eval_accuracy": final_metrics["accuracy"],
            "final_eval_f1_macro": final_metrics["f1_macro"],
            "elapsed_seconds": elapsed_seconds,
        }
    finally:
        wandb.unwatch(model) if "model" in locals() else None
        run.finish()

## Run 5 baseline experiments

In [11]:
def run_baseline_experiments(configs):
    results = []
    for index, config in enumerate(configs, start=1):
        print(f"\nRunning baseline {index}/{len(configs)}: {config['model_name']}")
        results.append(run_experiment(config, group="mnist-5-baselines", job_type="baseline"))

    if pd is not None:
        return pd.DataFrame(results).sort_values("best_eval_accuracy", ascending=False)
    return results


RUN_FIVE_BASELINES = True

baseline_results = run_baseline_experiments(BASELINE_CONFIGS) if RUN_FIVE_BASELINES else []
baseline_results


Running baseline 1/5: mlp_small


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


100%|██████████| 9.91M/9.91M [00:03<00:00, 3.01MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 102kB/s]
100%|██████████| 1.65M/1.65M [00:44<00:00, 36.9kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 22.2MB/s]


mlp_small-adam-lr1e-03-bs128-drop0.00 | epoch 1/10 | train_acc=0.8541 | eval_acc=0.9135 | eval_loss=0.2948
mlp_small-adam-lr1e-03-bs128-drop0.00 | epoch 2/10 | train_acc=0.9298 | eval_acc=0.9305 | eval_loss=0.2358
mlp_small-adam-lr1e-03-bs128-drop0.00 | epoch 3/10 | train_acc=0.9520 | eval_acc=0.9345 | eval_loss=0.2006
mlp_small-adam-lr1e-03-bs128-drop0.00 | epoch 4/10 | train_acc=0.9665 | eval_acc=0.9475 | eval_loss=0.1769
mlp_small-adam-lr1e-03-bs128-drop0.00 | epoch 5/10 | train_acc=0.9737 | eval_acc=0.9490 | eval_loss=0.1597
mlp_small-adam-lr1e-03-bs128-drop0.00 | epoch 6/10 | train_acc=0.9822 | eval_acc=0.9580 | eval_loss=0.1429
mlp_small-adam-lr1e-03-bs128-drop0.00 | epoch 7/10 | train_acc=0.9882 | eval_acc=0.9550 | eval_loss=0.1517
mlp_small-adam-lr1e-03-bs128-drop0.00 | epoch 8/10 | train_acc=0.9901 | eval_acc=0.9605 | eval_loss=0.1341
mlp_small-adam-lr1e-03-bs128-drop0.00 | epoch 9/10 | train_acc=0.9936 | eval_acc=0.9610 | eval_loss=0.1288
mlp_small-adam-lr1e-03-bs128-drop0.00

batch/accuracy,▁▇██████████████████
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,█▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch,▁▂▃▃▄▅▆▆▇██
eval/accuracy,▁▄▄▆▆█▇███
eval/best_accuracy,▁▄▄▆▆█████
eval/f1_macro,▁▄▄▆▆█▇███
eval/final_accuracy,▁
eval/final_f1_macro,▁
+6,...


...

batchnorm_cnn-sgd-lr1e-02-bs128-drop0.15 | epoch 1/10 | train_acc=0.3102 | eval_acc=0.3780 | eval_loss=1.7423
batchnorm_cnn-sgd-lr1e-02-bs128-drop0.15 | epoch 2/10 | train_acc=0.4773 | eval_acc=0.5405 | eval_loss=1.4330
batchnorm_cnn-sgd-lr1e-02-bs128-drop0.15 | epoch 3/10 | train_acc=0.5761 | eval_acc=0.6740 | eval_loss=1.1658
batchnorm_cnn-sgd-lr1e-02-bs128-drop0.15 | epoch 4/10 | train_acc=0.6447 | eval_acc=0.7775 | eval_loss=1.0197
batchnorm_cnn-sgd-lr1e-02-bs128-drop0.15 | epoch 5/10 | train_acc=0.7140 | eval_acc=0.8350 | eval_loss=0.8516
batchnorm_cnn-sgd-lr1e-02-bs128-drop0.15 | epoch 6/10 | train_acc=0.7592 | eval_acc=0.8125 | eval_loss=0.7634
batchnorm_cnn-sgd-lr1e-02-bs128-drop0.15 | epoch 7/10 | train_acc=0.8035 | eval_acc=0.8895 | eval_loss=0.5888
batchnorm_cnn-sgd-lr1e-02-bs128-drop0.15 | epoch 8/10 | train_acc=0.8377 | eval_acc=0.9080 | eval_loss=0.4833
batchnorm_cnn-sgd-lr1e-02-bs128-drop0.15 | epoch 9/10 | train_acc=0.8633 | eval_acc=0.9000 | eval_loss=0.4528
batchnorm_

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING Artifact "source-mnist-wandb-evaluation-v1-d__University_Projects_Individual_projects_Research-training_notebooks_WandB_2_evaluation.ipynb" already exists with the same content. No new version will be created.


batch/accuracy,▁▂▃▄▄▅▅▅▇▆▆▇▆▇▇███▇█
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,█▇▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
batch_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch,▁▂▃▃▄▅▆▆▇██
eval/accuracy,▁▃▅▆▇▇████
eval/best_accuracy,▁▃▅▆▇▇████
eval/f1_macro,▁▃▅▆▇▇████
eval/final_accuracy,▁
eval/final_f1_macro,▁
+6,...


,run_id,run_name,model_name,optimizer,learning_rate,batch_size,dropout,best_eval_accuracy,final_eval_accuracy,final_eval_f1_macro,elapsed_seconds
3,f9q8cc5h,dropout_cnn-adam-lr1e-03-bs128-drop0.35,dropout_cnn,adam,0.0010,128,0.35,0.9895,0.9875,0.987388,35.137989
2,0oummgc5,tiny_cnn-adam-lr1e-03-bs128-drop0.00,tiny_cnn,adam,0.0010,128,0.00,0.9810,0.9805,0.980641,32.740124
1,dybtxjzl,mlp_deep-adamw-lr3e-04-bs128-drop0.25,mlp_deep,adamw,0.0003,128,0.25,0.9620,0.9620,0.961559,29.544520
0,yusa9qmz,mlp_small-adam-lr1e-03-bs128-drop0.00,mlp_small,adam,0.0010,128,0.00,0.9610,0.9605,0.960455,29.662921
4,dh4638hy,batchnorm_cnn-sgd-lr1e-02-bs128-drop0.15,batchnorm_cnn,sgd,0.0100,128,0.15,0.9285,0.9285,0.928098,36.400888


## W&B Sweep

A Sweep lets W&B launch many runs with different hyperparameters. Turn on the last cell by setting `RUN_SWEEP = True`.

In [12]:
SWEEP_CONFIG = {
    "method": "bayes",
    "metric": {
        "name": "eval/accuracy",
        "goal": "maximize",
    },
    "parameters": {
        "model_name": {"values": list(MODEL_BUILDERS.keys())},
        "epochs": {"value": 3},
        "batch_size": {"values": [64, 128, 256]},
        "learning_rate": {"values": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]},
        "optimizer": {"values": ["adam", "adamw", "sgd"]},
        "dropout": {"values": [0.0, 0.15, 0.25, 0.35, 0.5]},
        "weight_decay": {"values": [0.0, 1e-5, 1e-4]},
        "momentum": {"values": [0.0, 0.9]},
        "train_subset": {"value": 12000},
        "test_subset": {"value": 2000},
        "log_every_n_batches": {"value": 50},
        "num_workers": {"value": 0},
        "seed": {"values": [42, 123, 2026]},
    },
}


def sweep_train():
    run_experiment(group="mnist-sweep", job_type="sweep", from_sweep=True)


def create_sweep():
    return wandb.sweep(SWEEP_CONFIG, project=PROJECT_NAME, entity=ENTITY)

In [ ]:
# RUN_SWEEP = False
RUN_SWEEP = True
SWEEP_COUNT = 10

if RUN_SWEEP:
    sweep_id = create_sweep()
    print(f"Sweep ID: {sweep_id}")
    wandb.agent(sweep_id, function=sweep_train, count=SWEEP_COUNT)

Create sweep with ID: 2lrds5ux
Sweep URL: https://wandb.ai/zintomfromnowhere/mnist-wandb-evaluation-v1/sweeps/2lrds5ux
Sweep ID: 2lrds5ux


wandb: Agent Starting Run: qsvy3aqo with config:
wandb: 	batch_size: 256
wandb: 	dropout: 0
wandb: 	epochs: 3
wandb: 	learning_rate: 0.001
wandb: 	log_every_n_batches: 50
wandb: 	model_name: mlp_small
wandb: 	momentum: 0
wandb: 	num_workers: 0
wandb: 	optimizer: adam
wandb: 	seed: 123
wandb: 	test_subset: 2000
wandb: 	train_subset: 12000
wandb: 	weight_decay: 1e-05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Mario\_netrc.


mlp_small-adam-lr1e-03-bs256-drop0.00 | epoch 1/3 | train_acc=0.8101 | eval_acc=0.8960 | eval_loss=0.3428
